In [7]:
import pycolmap
import numpy as np
import torch

sfm_path = "../data/NeRF/nerf_synthetic/chair"
reconstruction = pycolmap.Reconstruction(f"{sfm_path}/sparse/0")

print("Summary:")
print(reconstruction.summary())

Summary:
Reconstruction:
	num_rigs = 1
	num_cameras = 1
	num_frames = 100
	num_reg_frames = 100
	num_images = 100
	num_points3D = 14116
	num_observations = 70728
	mean_track_length = 5.01048
	mean_observations_per_image = 707.28
	mean_reprojection_error = 0.471156


## Camera

In [8]:
camera = reconstruction.cameras[1]
camera_width = camera.width
camera_height = camera.height
[camera_fx, camera_fy, camera_cx, camera_cy] = camera.params
# 相机内参
print(camera_width, camera_height, camera_fx, camera_fy, camera_cx, camera_cy)

800 800 1111.1110311937682 1111.1110311937682 400.0 400.0


## IMAGES

In [9]:
list_view_matrix = []
the_image = None
for image_id, image in reconstruction.images.items():
    # print(image_id, image.name)
    cam_from_world = image.cam_from_world()
    R = cam_from_world.rotation.matrix()  # 旋转矩阵
    t = cam_from_world.translation  # 平移向量
    W = np.eye(4)
    W[0:3, 0:3] = R
    W[0:3, 3] = t
    the_image = image
    list_view_matrix.append(W)  # 视图矩阵
view_matrices = np.stack(list_view_matrix)  # 视图矩阵
view_matrices.shape
the_image

Image(image_id=100, camera_id=1, frame_id=100, name="r_99.png", has_pose=1, triangulated=1238/2523)

In [10]:
type(the_image)
image_path = f"{sfm_path}/images/{image.name}"
image_path

'../data/NeRF/nerf_synthetic/chair/images/r_99.png'

## Point3D

In [11]:
print("Point3D:")
point3d_pos = np.array([])
npoint3d = len(reconstruction.points3D)
point3d_pos = np.zeros((npoint3d, 3))  # 稀疏点云的位置，世界坐标系下
point3d_rgb = np.zeros((npoint3d, 3))  # 稀疏点云的颜色0-1
for i, (point3D_id, point3D) in enumerate(reconstruction.points3D.items()):
    point3d_pos[i, :] = point3D.xyz
    point3d_rgb[i, :] = point3D.color / 255
point3d_rgb

Point3D:


array([[0.23921569, 0.4       , 0.18431373],
       [0.21960784, 0.40784314, 0.18823529],
       [0.71372549, 0.61960784, 0.4745098 ],
       ...,
       [0.78431373, 0.78039216, 0.54509804],
       [0.41960784, 0.58431373, 0.26666667],
       [0.99607843, 0.99607843, 0.98039216]], shape=(14116, 3))